<a href="https://colab.research.google.com/github/mr-zero-000/Statistical-Learning-e23034/blob/main/Assignment%205/Part_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

# --- DO NOT MODIFY: Synthetic Strain Sensor Data Generator ---
np.random.seed(42)
n_samples = 5000
n_features = 3

# Generating base stationary multivariate normal noise
base_data = np.random.multivariate_normal(
    mean=[0.5, -0.2, 1.1],
    cov=[[0.09, 0.02, 0.01],
         [0.02, 0.06, 0.03],
         [0.01, 0.03, 0.05]],
    size=n_samples
)

# Injecting a subtle, hidden structural drift in the final 1,000 snapshots
base_data[4000:, 0] += 0.015  # Sensor 1 drift
base_data[4000:, 2] -= 0.010  # Sensor 3 drift

df_strain = pd.DataFrame(base_data, columns=['Strain_Ch1', 'Strain_Ch2', 'Strain_Ch3'])
# ----------------------------------------------------------------

def verify_first_moment_homogeneity(df: pd.DataFrame, g_chunks: int = 5) -> dict:
    """
    Partitions the dataset into g chunks and evaluates first-moment homogeneity
    via Wilks' Lambda and Bartlett's Chi-Square approximation.
    """

    import numpy as np
    from scipy.stats import chi2

    # ----------------------------------------------------
    # 1. Basic dimensions
    # ----------------------------------------------------
    X = df.values
    n, m = X.shape

    if n % g_chunks != 0:
        raise ValueError("Dataset size must be divisible by g_chunks.")

    chunk_size = n // g_chunks

    # ----------------------------------------------------
    # 2. Global mean
    # ----------------------------------------------------
    grand_mean = np.mean(X, axis=0)

    # ----------------------------------------------------
    # 3. Compute W and B matrices
    # ----------------------------------------------------
    W = np.zeros((m, m))
    B = np.zeros((m, m))

    for i in range(g_chunks):

        start = i * chunk_size
        end = (i + 1) * chunk_size

        Xi = X[start:end]

        mean_i = np.mean(Xi, axis=0)

        # Within-group SSCP matrix
        centered = Xi - mean_i
        W += centered.T @ centered

        # Between-group SSCP matrix
        mean_diff = (mean_i - grand_mean).reshape(m, 1)
        B += chunk_size * (mean_diff @ mean_diff.T)

    # ----------------------------------------------------
    # 4. Wilks' Lambda
    # ----------------------------------------------------
    det_W = np.linalg.det(W)
    det_T = np.linalg.det(W + B)

    wilks_lambda = det_W / det_T

    # ----------------------------------------------------
    # 5. Bartlett Chi-Square Approximation
    # ----------------------------------------------------
    chi2_calc = -(
        n - 1 - (m + g_chunks) / 2
    ) * np.log(wilks_lambda)

    df_chi2 = m * (g_chunks - 1)

    p_value = 1 - chi2.cdf(chi2_calc, df=df_chi2)

    # ----------------------------------------------------
    # 6. Decision
    # ----------------------------------------------------
    alpha = 0.05

    if p_value < alpha:
        conclusion = (
            "Reject H0: First-moment homogeneity violated. "
            "The baseline center has shifted over time."
        )
    else:
        conclusion = (
            "Fail to reject H0: No statistically significant shift "
            "in the baseline center. First moment is homogeneous."
        )

    return {
        "Wilks_Lambda": wilks_lambda,
        "Bartlett_Chi2": chi2_calc,
        "Degrees_of_Freedom": df_chi2,
        "p_value": p_value,
        "Conclusion": conclusion
    }
results = verify_first_moment_homogeneity(df_strain, g_chunks=5)

for k, v in results.items():
    print(f"{k}: {v}")

Wilks_Lambda: 0.9904519796929591
Bartlett_Chi2: 47.9215049961488
Degrees_of_Freedom: 12
p_value: 3.2255259508895406e-06
Conclusion: Reject H0: First-moment homogeneity violated. The baseline center has shifted over time.
